# REPSOL Model Training (70/15/15)

This notebook runs the full pipeline:
1. Configure training hyperparameters.
2. Verify spectrogram tensor files in train/val/test.
3. Train EfficientNet with live progress bars.
4. Evaluate on validation and test sets.
5. Print detailed classification report and confusion matrix.

In [6]:
from pathlib import Path
import torch

# ===== Hyperparameters (edit these) =====
BATCH_SIZE = 16
EPOCHS = 15
LEARNING_RATE = 1e-3
PATIENCE = 4
MODEL_NAME = "efficientnet"

# ===== Paths =====
PROJECT_ROOT = Path(r"C:\home\ben\REPSOL")
SPECTROGRAM_DIR = PROJECT_ROOT / "Data" / "Spectrograms"
CHECKPOINT_PATH = PROJECT_ROOT / f"{MODEL_NAME}_best.pth"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SPECTROGRAM_DIR:", SPECTROGRAM_DIR)
print("CHECKPOINT_PATH:", CHECKPOINT_PATH)
print("DEVICE:", DEVICE)
print("BATCH_SIZE:", BATCH_SIZE, "| EPOCHS:", EPOCHS, "| LR:", LEARNING_RATE)

PROJECT_ROOT: C:\home\ben\REPSOL
SPECTROGRAM_DIR: C:\home\ben\REPSOL\Data\Spectrograms
CHECKPOINT_PATH: C:\home\ben\REPSOL\efficientnet_best.pth
DEVICE: cpu
BATCH_SIZE: 16 | EPOCHS: 15 | LR: 0.001


In [7]:
def count_pt_files(root):
    counts = {}
    for split in ["train", "val", "test"]:
        split_dir = root / split
        counts[split] = sum(1 for _ in split_dir.rglob("*.pt")) if split_dir.exists() else 0
    return counts

counts = count_pt_files(SPECTROGRAM_DIR)
print("PT files by split:", counts)
print("Total:", sum(counts.values()))

assert counts["train"] > 0, "No train .pt files found."
assert counts["val"] > 0, "No val .pt files found."
assert counts["test"] > 0, "No test .pt files found."

PT files by split: {'train': 1383, 'val': 296, 'test': 297}
Total: 1976


In [ ]:
# Install/verify training dependencies in the active notebook kernel
import importlib
import subprocess
import sys

required = ["torch", "torchvision", "torchaudio", "scikit-learn", "pandas", "tqdm", "numpy"]
name_map = {
    "scikit-learn": "sklearn",
}

for pkg in required:
    import_name = name_map.get(pkg, pkg.replace("-", "_"))
    try:
        importlib.import_module(import_name)
        print(f"OK: {pkg}")
    except Exception:
        print(f"Installing: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        print(f"Installed: {pkg}")

In [ ]:
import importlib
import torch

try:
    import torchvision  # noqa: F401
except Exception as e:
    raise RuntimeError(
        "torchvision is not available in this notebook kernel. "
        "Run the dependency cell right above this one, then retry."
    ) from e

import src.train as train_module

train_module = importlib.reload(train_module)
Trainer = train_module.Trainer

trainer = Trainer(
    spectrogram_dir=SPECTROGRAM_DIR,
    checkpoint_path=CHECKPOINT_PATH,
    model_name=MODEL_NAME,
    batch_size=BATCH_SIZE,
    max_epochs=EPOCHS,
    patience=PATIENCE,
    lr=LEARNING_RATE,
    device=DEVICE,
)

trainer.fit()

ModuleNotFoundError: No module named 'torchvision'

In [ ]:
import importlib
import torch
import src.evaluate as eval_module

eval_module = importlib.reload(eval_module)
evaluate_model = eval_module.evaluate_model

state = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
trainer.model.load_state_dict(state)

val_metrics = evaluate_model(trainer.model, trainer.val_loader, DEVICE)
test_metrics = evaluate_model(trainer.model, trainer.test_loader, DEVICE)

print("Validation Metrics")
print({
    "accuracy": round(val_metrics["accuracy"], 4),
    "precision": round(val_metrics["precision"], 4),
    "recall": round(val_metrics["recall"], 4),
    "f1": round(val_metrics["f1"], 4),
})

print("\nTest Metrics")
print({
    "accuracy": round(test_metrics["accuracy"], 4),
    "precision": round(test_metrics["precision"], 4),
    "recall": round(test_metrics["recall"], 4),
    "f1": round(test_metrics["f1"], 4),
})

In [ ]:
print("Test Classification Report:\n")
print(test_metrics["report"])

print("Test Confusion Matrix:")
print(test_metrics["confusion_matrix"])